In [30]:
import polars as pl

In [31]:
master = pl.read_csv("Store39_Dataset/MasterDataset.csv", try_parse_dates=True)
master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price
date,i64,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0,0,0,0,0,49.07
2016-06-02,103501,8.0,0,"""CLEANING""",3008,0,0,1284,0,0,0,0,49.14
2016-06-03,103501,10.0,0,"""CLEANING""",3008,0,0,1317,0,0,0,0,48.69
2016-06-04,103501,14.0,0,"""CLEANING""",3008,0,0,1820,0,0,0,0,48.69
2016-06-05,103501,11.0,0,"""CLEANING""",3008,0,0,1653,0,0,0,0,48.69


In [32]:
master = master.with_columns(
    [
        pl.col('unit_sales').log1p(),
        pl.col('transactions').log1p()
    ]
)

In [33]:
master = master.with_columns([
    pl.col("date").dt.weekday().cast(pl.Int8).alias("day_of_week"),
    (pl.col("date").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
    (pl.col("date").dt.day() <= 3).cast(pl.Int8).alias("is_month_start"),
    (pl.col("date").dt.day() >= 28).cast(pl.Int8).alias("is_month_end")
])
master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0


In [34]:
family_sales = master.select(["date", "unit_sales", "family"])
family_sales.head()

date,unit_sales,family
date,f64,str
2016-06-01,0.0,"""CLEANING"""
2016-06-02,2.197225,"""CLEANING"""
2016-06-03,2.397895,"""CLEANING"""
2016-06-04,2.70805,"""CLEANING"""
2016-06-05,2.484907,"""CLEANING"""


In [35]:
family_sales = family_sales.group_by(["family", "date"]).agg(pl.col("unit_sales").sum()).sort(by=["family", "date"])
family_sales.head()

family,date,unit_sales
str,date,f64
"""AUTOMOTIVE""",2016-06-01,0.0
"""AUTOMOTIVE""",2016-06-02,5.375278
"""AUTOMOTIVE""",2016-06-03,3.465736
"""AUTOMOTIVE""",2016-06-04,3.688879
"""AUTOMOTIVE""",2016-06-05,7.860185


In [36]:
family_sales = family_sales.with_columns(
    (pl.col("unit_sales").cum_sum() / pl.col("unit_sales").cum_count()).over("family").alias("family_mean_sales")
)
family_sales = family_sales.drop("unit_sales")
family_sales.head()

family,date,family_mean_sales
str,date,f64
"""AUTOMOTIVE""",2016-06-01,0.0
"""AUTOMOTIVE""",2016-06-02,2.687639
"""AUTOMOTIVE""",2016-06-03,2.947005
"""AUTOMOTIVE""",2016-06-04,3.132473
"""AUTOMOTIVE""",2016-06-05,4.078016


In [37]:
master = master.with_columns([
    (pl.col("unit_sales").cum_sum()/pl.col("unit_sales").cum_count()).over("item_nbr").alias("item_mean_sales"),
    pl.col("unit_sales").cumulative_eval(pl.element().std(ddof=0)).over("item_nbr").alias("item_std_sales")
])

master = master.join(family_sales, on=["family", "date"], how="left")

master = master.with_columns(
    (pl.col("item_mean_sales") / pl.col("family_mean_sales").clip(lower_bound=0.01)).alias("sales_vs_family_mean")
)

master = master.drop("family_mean_sales")

master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491


In [38]:
master = master.with_columns([
    pl.col("unit_sales").shift(0).over("item_nbr").alias("lag_1"),
    pl.col("unit_sales").shift(2).over("item_nbr").alias("lag_3"),
    pl.col("unit_sales").shift(4).over("item_nbr").alias("lag_5"),
    pl.col("unit_sales").shift(6).over("item_nbr").alias("lag_7"),
    pl.col("unit_sales").shift(13).over("item_nbr").alias("lag_14"),
    pl.col("unit_sales").shift(29).over("item_nbr").alias("lag_30"),
    pl.col("unit_sales").shift(59).over("item_nbr").alias("lag_60"),
    pl.col("unit_sales").shift(139).over("item_nbr").alias("lag_140")
])
master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null


In [39]:
master = master.with_columns([
    pl.col("unit_sales").rolling_mean(3).over("item_nbr").alias("rolling_mean_3"),
    pl.col("unit_sales").rolling_mean(7).over("item_nbr").alias("rolling_mean_7"),
    pl.col("unit_sales").rolling_mean(14).over("item_nbr").alias("rolling_mean_14"),
    pl.col("unit_sales").rolling_mean(30).over("item_nbr").alias("rolling_mean_30"),
    pl.col("unit_sales").rolling_mean(140).over("item_nbr").alias("rolling_mean_140"),
    pl.col("unit_sales").rolling_std(7).over("item_nbr").alias("rolling_std_7"),
    pl.col("unit_sales").rolling_std(14).over("item_nbr").alias("rolling_std_14"),
    pl.col("unit_sales").rolling_std(30).over("item_nbr").alias("rolling_std_30"),
    pl.col("unit_sales").rolling_max(7).over("item_nbr").alias("rolling_max_7"),
    pl.col("unit_sales").rolling_min(7).over("item_nbr").alias("rolling_min_7")
])
master.head(10)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null,1.531707,null,null,null,null,null,null,null,null,null
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null,2.43439,null,null,null,null,null,null,null,null,null
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null,2.530284,null,null,null,null,null,null,null,null,null
2016-06-06,103501,1.609438,0,"""CLEANING""",3008,0,0,7.196687,0,0,0,0,49.71,1,0,0,0,1.899586,0.915217,0.005206,1.609438,2.70805,2.197225,null,null,null,null,null,2.267465,null,null,null,null,null,null,null,null,null
2016-06-07,103501,2.197225,0,"""CLEANING""",3008,0,0,7.21891,0,0,0,0,50.37,2,0,0,0,1.942106,0.853703,0.005275,2.197225,2.484907,2.397895,0.0,null,null,null,null,2.09719,1.942106,null,null,null,0.922105,null,null,2.70805,0.0
2016-06-08,103501,0.0,0,"""CLEANING""",3008,0,0,7.171657,0,0,0,0,51.23,3,0,0,0,1.699342,1.024815,0.004643,0.0,1.609438,2.70805,2.197225,null,null,null,null,1.268887,1.942106,null,null,null,0.922105,null,null,2.70805,0.0
2016-06-09,103501,1.791759,0,"""CLEANING""",3008,0,0,7.138073,0,0,0,0,50.52,4,0,0,0,1.709611,0.966641,0.004682,1.791759,2.197225,2.484907,2.397895,null,null,null,null,1.329661,1.884182,null,null,null,0.916124,null,null,2.70805,0.0


In [40]:
master = master.with_columns([
    pl.col("transactions").shift(0).over("item_nbr").alias("transactions_lag_1"),
    pl.col("transactions").shift(2).over("item_nbr").alias("transactions_lag_3"),
    pl.col("transactions").shift(6).over("item_nbr").alias("transactions_lag_7"),
    pl.col("transactions").shift(13).over("item_nbr").alias("transactions_lag_14"),
    pl.col("transactions").rolling_mean(7).over("item_nbr").alias("transactions_rolling_7"),
    pl.col("transactions").rolling_mean(14).over("item_nbr").alias("transactions_rolling_14"),
    pl.col("transactions").rolling_mean(30).over("item_nbr").alias("transactions_rolling_30")
]).with_columns(
    (pl.col("transactions_lag_1")/pl.col("transactions_rolling_30")).clip(lower_bound=0.5).alias("traffic_ratio")
)
master.head(10)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,7.158514,null,null,null,null,null,null,null
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null,1.531707,null,null,null,null,null,null,null,null,null,7.183871,0.0,null,null,null,null,null,null
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null,2.43439,null,null,null,null,null,null,null,null,null,7.507141,7.158514,null,null,null,null,null,null
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null,2.530284,null,null,null,null,null,null,null,null,null,7.410952,7.183871,null,null,null,null,null,null
2016-06-06,103501,1.609438,0,"""CLEANING""",3008,0,0,7.196687,0,0,0,0,49.71,1,0,0,0,1.899586,0.915217,0.005206,1.609438,2.70805,2.197225,null,null,null,null,null,2.267465,null,null,null,null,null,null,null,null,null,7.196687,7.507141,null,null,null,null,null,null
2016-06-07,103501,2.197225,0,"""CLEANING""",3008,0,0,7.21891,0,0,0,0,50.37,2,0,0,0,1.942106,0.853703,0.005275,2.197225,2.484907,2.397895,0.0,null,null,null,null,2.09719,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.21891,7.410952,0.0,null,6.239439,null,null,null
2016-06-08,103501,0.0,0,"""CLEANING""",3008,0,0,7.171657,0,0,0,0,51.23,3,0,0,0,1.699342,1.024815,0.004643,0.0,1.609438,2.70805,2.197225,null,null,null,null,1.268887,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.171657,7.196687,7.158514,null,7.263962,null,null,null
2016-06-09,103501,1.791759,0,"""CLEANING""",3008,0,0,7.138073,0,0,0,0,50.52,4,0,0,0,1.709611,0.966641,0.004682,1.791759,2.197225,2.484907,2.397895,null,null,null,null,1.329661,1.884182,null,null,null,0.916124,null,null,2.70805,0.0,7.138073,7.21891,7.183871,null,7.261041,null,null,null


In [41]:
master = master.with_columns([
    pl.col("oil_price").shift(0).over("item_nbr").alias("oil_price_lag_1"),
    pl.col("oil_price").shift(2).over("item_nbr").alias("oil_price_lag_3"),
    pl.col("oil_price").shift(6).over("item_nbr").alias("oil_price_lag_7"),
    pl.col("oil_price").rolling_mean(7).over("item_nbr").alias("oil_price_rolling_mean_7"),
    pl.col("oil_price").rolling_mean(30).over("item_nbr").alias("oil_price_rolling_mean_30")
]).with_columns(
    ((pl.col("oil_price_lag_1") - pl.col("oil_price_lag_7")) / pl.col("oil_price_lag_7")).alias("oil_price_change_7")
)
master.head(10)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,49.07,null,null,null,null,null
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,7.158514,null,null,null,null,null,null,null,49.14,null,null,null,null,null
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null,1.531707,null,null,null,null,null,null,null,null,null,7.183871,0.0,null,null,null,null,null,null,48.69,49.07,null,null,null,null
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null,2.43439,null,null,null,null,null,null,null,null,null,7.507141,7.158514,null,null,null,null,null,null,48.69,49.14,null,null,null,null
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null,2.530284,null,null,null,null,null,null,null,null,null,7.410952,7.183871,null,null,null,null,null,null,48.69,48.69,null,null,null,null
2016-06-06,103501,1.609438,0,"""CLEANING""",3008,0,0,7.196687,0,0,0,0,49.71,1,0,0,0,1.899586,0.915217,0.005206,1.609438,2.70805,2.197225,null,null,null,null,null,2.267465,null,null,null,null,null,null,null,null,null,7.196687,7.507141,null,null,null,null,null,null,49.71,48.69,null,null,null,null
2016-06-07,103501,2.197225,0,"""CLEANING""",3008,0,0,7.21891,0,0,0,0,50.37,2,0,0,0,1.942106,0.853703,0.005275,2.197225,2.484907,2.397895,0.0,null,null,null,null,2.09719,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.21891,7.410952,0.0,null,6.239439,null,null,null,50.37,48.69,49.07,49.194286,null,0.026493
2016-06-08,103501,0.0,0,"""CLEANING""",3008,0,0,7.171657,0,0,0,0,51.23,3,0,0,0,1.699342,1.024815,0.004643,0.0,1.609438,2.70805,2.197225,null,null,null,null,1.268887,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.171657,7.196687,7.158514,null,7.263962,null,null,null,51.23,49.71,49.14,49.502857,null,0.042532
2016-06-09,103501,1.791759,0,"""CLEANING""",3008,0,0,7.138073,0,0,0,0,50.52,4,0,0,0,1.709611,0.966641,0.004682,1.791759,2.197225,2.484907,2.397895,null,null,null,null,1.329661,1.884182,null,null,null,0.916124,null,null,2.70805,0.0,7.138073,7.21891,7.183871,null,7.261041,null,null,null,50.52,50.37,48.69,49.7,null,0.037585


In [42]:
master = master.with_columns([
    pl.col("onpromotion").shift(0).over("item_nbr").alias("promo_lag_1"),
    pl.col("onpromotion").shift(2).over("item_nbr").alias("promo_lag_3"),
    pl.col("onpromotion").shift(6).over("item_nbr").alias("promo_lag_7"),
    pl.col("onpromotion").cast(pl.Int8).rolling_sum(7).over("item_nbr").alias("promo_count_7"),
    pl.col("onpromotion").cast(pl.Int8).rolling_sum(30).over("item_nbr").alias("promo_count_30")
])
master = master.with_columns(
    pl.when(pl.col("onpromotion").over("item_nbr") == 1)
    .then(pl.col("date"))
    .otherwise(None)
    .forward_fill()
    .over("item_nbr")
    .alias("last_promo_date")
).with_columns(
    (pl.col("date") - pl.col("last_promo_date"))
    .dt.total_days()
    .fill_null(999)
    .alias("days_since_last_promo")
).drop("last_promo_date")

master.head(10)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,49.07,null,null,null,null,null,0,null,null,null,null,999
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,7.158514,null,null,null,null,null,null,null,49.14,null,null,null,null,null,0,null,null,null,null,0
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null,1.531707,null,null,null,null,null,null,null,null,null,7.183871,0.0,null,null,null,null,null,null,48.69,49.07,null,null,null,null,0,0,null,null,null,0
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null,2.43439,null,null,null,null,null,null,null,null,null,7.507141,7.158514,null,null,null,null,null,null,48.69,49.14,null,null,null,null,0,0,null,null,null,0
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null,2.530284,null,null,null,null,null,null,null,null,null,7.410952,7.183871,null,null,null,null,null,null,48.69,48.69,null,null,null,null,0,0,null,null,null,0
2016-06-06,103501,1.609438,0,"""CLEANING""",3008,0,0,7.196687,0,0,0,0,49.71,1,0,0,0,1.899586,0.915217,0.005206,1.609438,2.70805,2.197225,null,null,null,null,null,2.267465,null,null,null,null,null,null,null,null,null,7.196687,7.507141,null,null,null,null,null,null,49.71,48.69,null,null,null,null,0,0,null,null,null,999
2016-06-07,103501,2.197225,0,"""CLEANING""",3008,0,0,7.21891,0,0,0,0,50.37,2,0,0,0,1.942106,0.853703,0.005275,2.197225,2.484907,2.397895,0.0,null,null,null,null,2.09719,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.21891,7.410952,0.0,null,6.239439,null,null,null,50.37,48.69,49.07,49.194286,null,0.026493,0,0,0,0,null,0
2016-06-08,103501,0.0,0,"""CLEANING""",3008,0,0,7.171657,0,0,0,0,51.23,3,0,0,0,1.699342,1.024815,0.004643,0.0,1.609438,2.70805,2.197225,null,null,null,null,1.268887,1.942106,null,null,null,0.922105,null,null,2.70805,0.0,7.171657,7.196687,7.158514,null,7.263962,null,null,null,51.23,49.71,49.14,49.502857,null,0.042532,0,0,0,0,null,0
2016-06-09,103501,1.791759,0,"""CLEANING""",3008,0,0,7.138073,0,0,0,0,50.52,4,0,0,0,1.709611,0.966641,0.004682,1.791759,2.197225,2.484907,2.397895,null,null,null,null,1.329661,1.884182,null,null,null,0.916124,null,null,2.70805,0.0,7.138073,7.21891,7.183871,null,7.261041,null,null,null,50.52,50.37,48.69,49.7,null,0.037585,0,0,0,0,null,0


In [43]:
holidays = pl.read_csv("Store39_Dataset/holidays.csv", try_parse_dates=True)

holiday_dates = (
    holidays.filter(pl.col("is_Holiday") == 1).select("date").sort("date")["date"].to_list()
)

unique_dates = master["date"].unique().sort().to_list()

days_since = []
days_to = []

for d in unique_dates:
    past = [h for h in holiday_dates if h < d]
    days_since.append((d - past[-1]).days if past else 999)

    future = [h for h in holiday_dates if h > d]
    days_to.append((future[0] - d).days if future else 999)

holiday_distances = pl.DataFrame({
    "date": unique_dates,
    "days_since_last_holiday": days_since,
    "days_to_next_holiday": days_to,
})
holiday_distances.head()

date,days_since_last_holiday,days_to_next_holiday
date,i64,i64
2016-06-01,999,130
2016-06-02,999,129
2016-06-03,999,128
2016-06-04,999,127
2016-06-05,999,126


In [44]:
master = master.join(holiday_distances, on="date", how="left")
master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0.0,0,0,0,0,49.07,3,0,1,0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,49.07,null,null,null,null,null,0,null,null,null,null,999,999,130
2016-06-02,103501,2.197225,0,"""CLEANING""",3008,0,0,7.158514,0,0,0,0,49.14,4,0,1,0,1.098612,1.098612,0.005808,2.197225,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,7.158514,null,null,null,null,null,null,null,49.14,null,null,null,null,null,0,null,null,null,null,0,999,129
2016-06-03,103501,2.397895,0,"""CLEANING""",3008,0,0,7.183871,0,0,0,0,48.69,5,1,1,0,1.531707,1.086174,0.005793,2.397895,0.0,null,null,null,null,null,null,1.531707,null,null,null,null,null,null,null,null,null,7.183871,0.0,null,null,null,null,null,null,48.69,49.07,null,null,null,null,0,0,null,null,null,0,999,128
2016-06-04,103501,2.70805,0,"""CLEANING""",3008,0,0,7.507141,0,0,0,0,48.69,6,1,0,0,1.825793,1.069715,0.005652,2.70805,2.197225,null,null,null,null,null,null,2.43439,null,null,null,null,null,null,null,null,null,7.507141,7.158514,null,null,null,null,null,null,48.69,49.14,null,null,null,null,0,0,null,null,null,0,999,127
2016-06-05,103501,2.484907,0,"""CLEANING""",3008,0,0,7.410952,0,0,0,0,48.69,7,1,0,0,1.957615,0.992442,0.005491,2.484907,2.397895,0.0,null,null,null,null,null,2.530284,null,null,null,null,null,null,null,null,null,7.410952,7.183871,null,null,null,null,null,null,48.69,48.69,null,null,null,null,0,0,null,null,null,0,999,126


In [45]:
master = master.drop_nulls(subset=[
    "rolling_mean_140"
])
master.null_count()
master.head(10)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64
2016-10-18,103501,1.386294,1,"""CLEANING""",3008,0,0,7.118826,0,0,0,0,50.3,2,0,0,0,1.566688,0.634997,0.004117,1.386294,1.94591,1.386294,0.693147,1.386294,0.0,1.098612,0.0,1.647214,1.498117,1.33745,1.337711,1.566688,0.453934,0.692677,0.712919,2.079442,0.693147,7.118826,7.361375,6.976348,7.04316,7.189647,7.2012,7.196245,0.989242,50.3,50.35,50.14,50.275714,47.975,0.003191,1,1,1,7,8,0,9,15
2016-10-19,103501,1.609438,1,"""CLEANING""",3008,0,0,7.018402,0,0,0,0,51.59,3,0,0,0,1.566991,0.632752,0.004123,1.609438,1.609438,2.079442,1.386294,1.098612,1.791759,1.609438,2.197225,1.535057,1.629016,1.353389,1.391359,1.578184,0.283082,0.696445,0.667919,2.079442,1.386294,7.018402,7.177019,7.112327,7.021976,7.195654,7.199432,7.194218,0.975561,51.59,49.97,50.47,50.482857,48.25,0.022191,1,1,1,7,9,0,10,14
2016-10-20,103501,1.098612,1,"""CLEANING""",3008,0,0,7.106606,0,0,0,0,50.31,4,0,0,0,1.563693,0.631735,0.004119,1.098612,1.386294,1.94591,1.386294,0.0,0.0,1.94591,2.397895,1.364782,1.587918,1.353389,1.368254,1.570337,0.339461,0.696445,0.665575,2.079442,1.098612,7.106606,7.118826,7.133296,7.167038,7.194837,7.205476,7.189512,0.988468,50.31,50.3,50.35,50.46,48.465333,-0.000794,1,1,1,7,10,999,11,13
2016-10-21,103501,0.0,0,"""CLEANING""",3008,0,0,7.085901,0,0,0,0,50.61,5,1,0,0,1.552758,0.642867,0.004094,0.0,1.609438,1.609438,2.079442,1.94591,1.098612,1.609438,2.70805,0.902683,1.389876,1.353389,1.368254,1.553209,0.694945,0.696445,0.665575,2.079442,0.0,7.085901,7.018402,7.448334,7.473637,7.188066,7.199681,7.190561,0.985445,50.61,51.59,50.35,50.497143,48.641333,0.005164,0,1,1,6,10,999,12,12
2016-10-22,103501,2.484907,1,"""CLEANING""",3008,0,0,7.38461,0,0,0,0,50.61,6,1,0,0,1.559231,0.645291,0.004107,2.484907,1.098612,1.386294,1.94591,2.197225,1.098612,2.302585,2.484907,1.194506,1.4478,1.391889,1.414464,1.551615,0.77436,0.744929,0.693737,2.484907,0.0,7.38461,7.106606,7.361375,7.448916,7.178963,7.193322,7.199637,1.025692,50.61,50.31,50.35,50.534286,48.791667,0.005164,1,1,1,6,11,999,13,11
2016-10-23,103501,1.609438,1,"""CLEANING""",3008,0,0,7.314553,0,0,0,0,50.61,7,1,0,0,1.559578,0.643075,0.004105,1.609438,0.0,1.609438,1.609438,1.609438,1.791759,1.098612,1.609438,1.364782,1.399732,1.349904,1.431491,1.545362,0.748291,0.71188,0.691984,2.484907,0.0,7.314553,7.085901,7.177019,7.149917,7.172274,7.183725,7.206107,1.015049,50.61,50.61,49.97,50.571429,49.0,0.012808,1,0,1,6,12,999,14,10
2016-10-24,103501,1.386294,1,"""CLEANING""",3008,0,0,7.04316,0,0,0,0,50.18,1,0,0,0,1.558391,0.641028,0.004106,1.386294,2.484907,1.098612,1.386294,0.0,0.693147,1.098612,2.197225,1.82688,1.367855,1.333965,1.417976,1.543768,0.7426,0.70811,0.688657,2.484907,0.0,7.04316,7.38461,7.118826,7.184629,7.153151,7.176099,7.195734,0.978797,50.18,50.61,50.3,50.601429,49.194,-0.002386,1,1,1,6,13,999,15,9
2016-10-25,103501,1.098612,1,"""CLEANING""",3008,0,0,7.171657,0,0,0,0,49.45,2,0,0,0,1.555263,0.639961,0.004102,1

In [46]:
master.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [47]:
from datetime import date
train_start = date(2016, 10, 18)
train_end = date(2017, 6, 16)
val_start = date(2017, 6, 17)
val_end = date(2017, 7, 16)
test_start = date(2017, 7, 17)
test_end = date(2017, 8, 15)

In [48]:
Master_train = master.filter((pl.col("date") >= train_start) & (pl.col("date") <= train_end))
Master_val = master.filter((pl.col("date") >= val_start) & (pl.col("date") <= val_end))
Master_test = master.filter((pl.col("date") >= test_start) & (pl.col("date") <= test_end))
Master_full_train = master.filter((pl.col("date") >= train_start) & (pl.col("date") <= val_end))

In [49]:
Master_train.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [50]:
Master_val.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [51]:
Master_test.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [52]:
Master_full_train.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [53]:
len(Master_full_train)

837760

In [54]:
len(Master_train)

745360

In [55]:
len(Master_val)

92400

In [56]:
len(Master_test)

92400

In [57]:
Master_train.write_csv("Store39_Dataset/MasterTrain.csv")
Master_val.write_csv("Store39_Dataset/MasterVal.csv")
Master_test.write_csv("Store39_Dataset/MasterTest.csv")
Master_full_train.write_csv("Store39_Dataset/MasterFullTrain.csv")

In [58]:
Master_train.head(200).write_csv("Store39_Dataset/MasterTrainPreview.csv")

In [59]:
Master_train.filter(pl.col('unit_sales') < 0)

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price,day_of_week,is_weekend,is_month_start,is_month_end,item_mean_sales,item_std_sales,sales_vs_family_mean,lag_1,lag_3,lag_5,lag_7,lag_14,lag_30,lag_60,lag_140,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_140,rolling_std_7,rolling_std_14,rolling_std_30,rolling_max_7,rolling_min_7,transactions_lag_1,transactions_lag_3,transactions_lag_7,transactions_lag_14,transactions_rolling_7,transactions_rolling_14,transactions_rolling_30,traffic_ratio,oil_price_lag_1,oil_price_lag_3,oil_price_lag_7,oil_price_rolling_mean_7,oil_price_rolling_mean_30,oil_price_change_7,promo_lag_1,promo_lag_3,promo_lag_7,promo_count_7,promo_count_30,days_since_last_promo,days_since_last_holiday,days_to_next_holiday
date,i64,f64,i64,str,i64,i64,i64,f64,i64,i64,i64,i64,f64,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64
